In [1]:
!pip install -U ultralytics --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.1 MB/s eta 0:00:00


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("PyTorch version:", torch.__version__)

from ultralytics import YOLO
import ultralytics
ultralytics.checks()

Ultralytics 8.4.142 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7035.6/8062.4 GB disk)


In [3]:
!wget https://github.com/ultralytics/yolov5/releases/download/v1.0/DOTAv1.zip -P /kaggle/working/data/
!unzip -q /kaggle/working/data/DOTAv1.zip -d /kaggle/working/data/
!ls /kaggle/working/data/DOTAv1

--2026-09-06 16:03:48--  https://github.com/ultralytics/yolov5/releases/download/v1.0/DOTAv1.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/264818686/31d36653-7214-41b0-bfe3-b66b4d39503b?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-09-06T16%3A50%3A56Z&rscd=attachment%3B+filename%3DDOTAv1.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-09-06T15%3A50%3A23Z&ske=2026-09-06T16%3A50%3A56Z&sks=b&skv=2018-11-09&sig=ooAZ11JgzSlYbapOsAWWcmtTn7nNd3ESsjoKo21K2k8%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4ODcxNDIyOCwibmJmIjoxNzg4NzEwNjI4LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvcmU

In [4]:
DOTA_CLASSES = [
    "plane", "ship", "storage-tank", "baseball-diamond", "tennis-court",
    "basketball-court", "ground-track-field", "harbor", "bridge",
    "large-vehicle", "small-vehicle", "helicopter", "roundabout",
    "soccer-ball-field", "swimming-pool"
]

import yaml
dota_yaml = {
    'path': '/kaggle/working/data/DOTAv1',
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: name for i, name in enumerate(DOTA_CLASSES)}
}
with open('/kaggle/working/dota.yaml', 'w') as f:
    yaml.dump(dota_yaml, f, sort_keys=False)
print(open('/kaggle/working/dota.yaml').read())

path: /kaggle/working/data/DOTAv1
train: images/train
val: images/val
names:
  0: plane
  1: ship
  2: storage-tank
  3: baseball-diamond
  4: tennis-court
  5: basketball-court
  6: ground-track-field
  7: harbor
  8: bridge
  9: large-vehicle
  10: small-vehicle
  11: helicopter
  12: roundabout
  13: soccer-ball-field
  14: swimming-pool



In [5]:
from ultralytics import YOLO
import shutil, os

def backup_dota_checkpoint(trainer):
    src = trainer.save_dir / 'weights' / 'last.pt'
    if src.exists():
        os.makedirs('/kaggle/working/checkpoint_backups', exist_ok=True)
        shutil.copy(src, '/kaggle/working/checkpoint_backups/dota_last_backup.pt')

model_dota = YOLO("yolo26n-obb.pt")
model_dota.add_callback("on_train_epoch_end", backup_dota_checkpoint)

model_dota.train(
    data='/kaggle/working/dota.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    project='/kaggle/working/runs', name='dota_full',
    seed=42, deterministic=True, exist_ok=True,
    patience=15,
    cache=True,
    workers=4,
    amp=True,
    save_period=5,
    time=6,             # comfortable cap given the fast 2.9min/epoch pace, leaves buffer
    max_det=1939,        # explicit -- matches the actual max object count, avoids silent recall capping
)

Ultralytics 8.4.142 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dota.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=1939, mixup=0.0, mode=train, model=yolo26n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dota_full, nbs=64, nms=None, opset=None, optimiz

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/148      7.34G      1.447      2.586    0.01155    0.04313        170        640: 100% ━━━━━━━━━━━━ 89/89 1.9it/s 47.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 3.3it/s 8.7s
                   all        458      28853      0.626       0.33      0.348      0.241

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss angle_loss  Instances       Size
      3/213      7.32G      1.427      1.985     0.0113     0.0441        216        640: 100% ━━━━━━━━━━━━ 89/89 1.8it/s 48.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 3.5it/s 8.2s
                   all        458      28853      0.694      0.396      0.401      0.276

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss angle_loss  Instances       Size
      4/248      7.32G      1.404      1.657     0.0113    0.04052         43        640: 100% ━━━━━━━━━━━━ 89/89 1.9it/s 47.6s
    

ultralytics.utils.metrics.OBBMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7da542cad700>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,   